In [2]:
import kagglehub
import pandas as pd
import numpy as np
import ast

In [3]:
path = kagglehub.dataset_download(
    "rounakbanik/the-movies-dataset"
)
metaDataDF = pd.read_csv(f"{path}/movies_metadata.csv")
ratingsDF = pd.read_csv(f"{path}/ratings_small.csv")


C:\Users\Johnny\AppData\Local\Temp\ipykernel_25524\3779643782.py:4: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  metaDataDF = pd.read_csv(f"{path}/movies_metadata.csv")


# Dataset - The Movies Dataset by Rounak Banik
report by John Lopes & Taha Riyaan

## Description

"About Dataset
Context
These files contain metadata for all 45,000 movies listed in the Full MovieLens Dataset. The dataset consists of movies released on or before July 2017. Data points include cast, crew, plot keywords, budget, revenue, posters, release dates, languages, production companies, countries, TMDB vote counts and vote averages.

This dataset also has files containing 26 million ratings from 270,000 users for all 45,000 movies. Ratings are on a scale of 1-5 and have been obtained from the official GroupLens website.

Content
This dataset consists of the following files:

movies_metadata.csv: The main Movies Metadata file. Contains information on 45,000 movies featured in the Full MovieLens dataset. Features include posters, backdrops, budget, revenue, release dates, languages, production countries and companies.

keywords.csv: Contains the movie plot keywords for our MovieLens movies. Available in the form of a stringified JSON Object.

credits.csv: Consists of Cast and Crew Information for all our movies. Available in the form of a stringified JSON Object.

links.csv: The file that contains the TMDB and IMDB IDs of all the movies featured in the Full MovieLens dataset.

links_small.csv: Contains the TMDB and IMDB IDs of a small subset of 9,000 movies of the Full Dataset.

ratings_small.csv: The subset of 100,000 ratings from 700 users on 9,000 movies.

The Full MovieLens Dataset consisting of 26 million ratings and 750,000 tag applications from 270,000 users on all the 45,000 movies in this dataset can be accessed here

Acknowledgements
This dataset is an ensemble of data collected from TMDB and GroupLens.
The Movie Details, Credits and Keywords have been collected from the TMDB Open API. This product uses the TMDb API but is not endorsed or certified by TMDb. Their API also provides access to data on many additional movies, actors and actresses, crew members, and TV shows. You can try it for yourself here.

The Movie Links and Ratings have been obtained from the Official GroupLens website. The files are a part of the dataset available here



Inspiration
This dataset was assembled as part of my second Capstone Project for Springboard's Data Science Career Track. I wanted to perform an extensive EDA on Movie Data to narrate the history and the story of Cinema and use this metadata in combination with MovieLens ratings to build various types of Recommender Systems.

Both my notebooks are available as kernels with this dataset: The Story of Film and Movie Recommender Systems

Some of the things you can do with this dataset:
Predicting movie revenue and/or movie success based on a certain metric. What movies tend to get higher vote counts and vote averages on TMDB? Building Content Based and Collaborative Filtering Based Recommendation Engines."

We decided to go with this dataset because we enjoying watching movies

## Code

### Cleaning

### Study 1

subset = genres, revenue, runtime, production_companies, title

Jaccard

In [3]:
def extractGenreNames(genres):
    if pd.isna(genres):
        return set()
    if isinstance(genres, str):
        try:
            genres = ast.literal_eval(genres)
        except (ValueError, SyntaxError):
            return set()
    if isinstance(genres, list):
        return {item['name'] for item in genres if isinstance(item, dict) and 'name' in item}
    return set()

def jaccardSimilarity(a, b):
    if not a and not b:
        return 1.0
    union = a | b
    return len(a & b) / len(union) if union else 0.0

movie = "Minions"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieGenres = extractGenreNames(movieRow.iloc[0]['genres']) if not movieRow.empty else set()
print(movie + " genres: " + str(movieGenres))

jaccardScores = []
for title in metaDataDF['title']:
    movieRow = metaDataDF[metaDataDF['title'] == title]
    if movieRow.empty:
        movieRow = metaDataDF[metaDataDF['original_title'] == title]
    genres = extractGenreNames(movieRow.iloc[0]['genres']) if not movieRow.empty else set()
    jaccardScores.append({
        'title': title,
        'genres': genres,
        'jaccard_similarity': jaccardSimilarity(movieGenres, genres),
    })

jaccardDF = pd.DataFrame(jaccardScores).sort_values('jaccard_similarity', ascending=False)
jaccardDF = jaccardDF[jaccardDF['title'] != movie]
jaccardDF.head(10)

Minions genres: {'Comedy', 'Family', 'Adventure', 'Animation'}


,title,genres,jaccard_similarity
8610,Asterix the Gaul,"{Comedy, Family, Adventure, Animation}",1.0
8626,Asterix and Cleopatra,"{Comedy, Family, Adventure, Animation}",1.0
43156,Smurfs: The Lost Village,"{Adventure, Family, Comedy, Animation}",1.0
608,The Aristocats,"{Family, Comedy, Adventure, Animation}",1.0
41586,Tri bogatyrya i Shamakhanskaya tsaritsa,"{Family, Comedy, Adventure, Animation}",1.0
40608,Storks,"{Adventure, Family, Comedy, Animation}",1.0
39750,VeggieTales: Tomato Sawyer & Huckleberry Larry...,"{Family, Comedy, Adventure, Animation}",1.0
39734,VeggieTales: Dave and the Giant Pickle,"{Adventure, Family, Comedy, Animation}",1.0
43294,Cars 3,"{Family, Comedy, Adventure, Animation}",1.0
39334,Ice Age: Collision Course,"{Adventure, Family, Comedy, Animation}",1.0


In [4]:
def extractCompanyNames(companies):
    if pd.isna(companies):
        return set()
    if isinstance(companies, str):
        try:
            companies = ast.literal_eval(companies)
        except (ValueError, SyntaxError):
            return set()
    if isinstance(companies, list):
        return {item['name'] for item in companies if isinstance(item, dict) and 'name' in item}
    return set()

movie = "Pocahontas"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieCompanies = extractCompanyNames(movieRow.iloc[0]['production_companies']) if not movieRow.empty else set()
print(movie + " production companies: " + str(movieCompanies))

jaccardScores = []
for title in metaDataDF['title']:
    movieRow = metaDataDF[metaDataDF['title'] == title]
    if movieRow.empty:
        movieRow = metaDataDF[metaDataDF['original_title'] == title]
    companies = extractCompanyNames(movieRow.iloc[0]['production_companies']) if not movieRow.empty else set()
    jaccardScores.append({
        'title': title,
        'companies': companies,
        'jaccard_similarity': jaccardSimilarity(movieCompanies, companies),
    })

jaccardDF = pd.DataFrame(jaccardScores).sort_values('jaccard_similarity', ascending=False)
jaccardDF = jaccardDF[jaccardDF['title'] != movie]
jaccardDF.head(10)

Pocahontas production companies: {'Walt Disney Pictures', 'Walt Disney Feature Animation'}


,title,companies,jaccard_similarity
1798,Mulan,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
5744,Treasure Planet,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
10520,Chicken Little,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
359,The Lion King,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
4238,Atlantis: The Lost Empire,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
5310,Lilo & Stitch,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
3493,Dinosaur,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
3891,The Emperor's New Groove,"{Walt Disney Pictures, Walt Disney Feature Ani...",1.000000
1980,The Rescuers Down Under,"{Walt Disney Pictures, Silver Screen Partners ...",0.666667
1520,Air Bud,{Walt Disney Pictures},0.500000


Manhattan

In [5]:
movie = "Mortal Kombat"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieRevenue = movieRow.iloc[0]['revenue'] if not movieRow.empty else 0.0
print(movie + " revenue: " + str(movieRevenue))

manhattanDistances = []
for i, row in metaDataDF.iterrows():
    rev = row['revenue']
    if pd.isna(rev) or pd.isna(movieRevenue):
        dist = float('inf')
    else:
        dist = abs(rev - movieRevenue)
    manhattanDistances.append({
        'title': row['title'],
        'revenue': rev,
        'manhattan_distance': dist,
    })

manhattanDF = pd.DataFrame(manhattanDistances)
manhattanDF = manhattanDF[manhattanDF['title'] != movie]
manhattanDF = manhattanDF.sort_values('manhattan_distance')
manhattanDF.head(10)

Mortal Kombat revenue: 122195920.0


,title,revenue,manhattan_distance
1885,Poltergeist,122200000.0,4080.0
14494,Invictus,122233971.0,38051.0
21604,Prisoners,122126687.0,69233.0
7011,Peter Pan,121975011.0,220909.0
1631,MouseHunt,122417389.0,221469.0
489,Executive Decision,121969216.0,226704.0
14234,Surrogates,122444772.0,248852.0
10762,Nanny McPhee,122489822.0,293902.0
31322,Magic Mike XXL,122513057.0,317137.0
5255,Spirit: Stallion of the Cimarron,122563539.0,367619.0


Euclidean

In [6]:
movie = "Avatar"
movieName = movie.lower()

movieRow = metaDataDF[metaDataDF['title'].str.lower() == movieName]
if movieRow.empty:
    movieRow = metaDataDF[metaDataDF['original_title'].str.lower() == movieName]

movieRuntime = movieRow.iloc[0]['runtime'] if not movieRow.empty else 0.0
print(movie + " runtime: " + str(movieRuntime))

euclideanDistances = []
for i, row in metaDataDF.iterrows():
    rt = row['runtime']
    if pd.isna(rt) or pd.isna(movieRuntime):
        dist = float('inf')
    else:
        dist = np.sqrt((rt - movieRuntime) ** 2)
    euclideanDistances.append({
        'title': row['title'],
        'runtime': rt,
        'euclidean_distance': dist,
    })

euclideanDF = pd.DataFrame(euclideanDistances)
euclideanDF = euclideanDF[euclideanDF['title'] != movie]
euclideanDF = euclideanDF.sort_values('euclidean_distance')
euclideanDF.head(10)

Avatar runtime: 162.0


,title,runtime,euclidean_distance
38801,Toni Erdmann,162.0,0.0
14992,Bunty Aur Babli,162.0,0.0
18451,The Disappearance of Haruhi Suzumiya,162.0,0.0
36560,Garv: Pride and Honour,162.0,0.0
33610,Times of Joy and Sorrow,162.0,0.0
29622,Sarfarosh,162.0,0.0
33434,A Tale of Two Cities,162.0,0.0
12520,Marketa Lazarová,162.0,0.0
7934,How the West Was Won,162.0,0.0
36571,Rakht,162.0,0.0


Edit Distance

In [ ]:
def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previousRow = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        currentRow = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previousRow[j + 1] + 1
            deletions = currentRow[j] + 1
            substitutions = previousRow[j] + (c1 != c2)
            currentRow.append(min(insertions, deletions, substitutions))
        previousRow = currentRow
    return previousRow[-1]

movie = "Pacific Rum"
editDistances = []
for title in metaDataDF['title']:
    dist = levenshtein(movie, str(title))
    editDistances.append({
        'title': title,
        'edit_distance': dist,
    })

editDF = pd.DataFrame(editDistances).sort_values('edit_distance')
editDF.head(10)

,title,edit_distance
21123,Pacific Rim,1
5132,Panic Room,5
26192,Sacrifice,6
38348,Sacrifice,6
18385,Sacrifice,6
40995,Pacific Banana,6
30770,Traffic Jam,6
9239,Mimic 2,7
11792,Maniac Cop,7
35942,Panic,7


### Study 2

### Study 3

### Study 4

In [4]:
user_item_matrix = ratingsDF.pivot(index='userId', columns='movieId', values='rating').fillna(0)
# user_item_matrix.head()
user_item_array = user_item_matrix.to_numpy()
user_item_array_validation = user_item_array.copy()
print(len(user_item_array))

671


In [5]:
# Remove 10% of ratings randomly and track affected users
np.random.seed(17)

# Calculate 10% of total ratings
num_to_remove = int(len(user_item_array) * 0.1)

# Randomly select indices to remove
remove_indices = np.random.choice(len(user_item_array), num_to_remove, replace=False)
# print(f"Indices to remove: {remove_indices}")

for idx in remove_indices:
    for i in range(len(user_item_array_validation[idx])):
        user_item_array_validation[idx][i] = 0

In [45]:
def matrix_factorization(R, P, Q, K, steps=500, alpha=0.0004, beta=0.02):
    '''
    R: rating matrix
    P: |U| * K (User features matrix)
    Q: |D| * K (Item features matrix)
    K: latent features
    steps: iterations
    alpha: learning rate
    beta: regularization parameter'''
    Q = Q.T
    error=[]
    for step in range(steps):
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    # calculate error
                    eij = R[i][j] - np.dot(P[i,:],Q[:,j])

                    for k in range(K):
                        # calculate gradient with a and beta parameter
                        P[i][k] = P[i][k] + alpha * (2 * eij * Q[k][j] - beta * P[i][k])
                        Q[k][j] = Q[k][j] + alpha * (2 * eij * P[i][k] - beta * Q[k][j])

        eR = np.dot(P,Q)

        e = 0

        for i in range(len(R)):

            for j in range(len(R[i])):

                if R[i][j] > 0:

                    e = e + pow(R[i][j] - np.dot(P[i,:],Q[:,j]), 2)

                    for k in range(K):

                        e = e + (beta/2) * (pow(P[i][k],2) + pow(Q[k][j],2))
        error.append(e)
        # 0.001: local minimum
        if e < 0.001:

            break
    for i in range(len(error)):
        print(error[i])    
    return P, Q.T

R = np.array(user_item_array_validation)
# N: num of User
N = len(R)
# M: num of Movie
M = len(R[0])
# Num of Features
K = 15
P = np.random.rand(N,K)
Q = np.random.rand(M,K)
nP, nQ = matrix_factorization(R, P, Q, K)
nR = np.dot(nP, nQ.T)

126794.39800846929
114439.39212953087
107439.65236034381
102663.39815494532
99103.37796613995
96303.2874114252
94018.68889372401
92104.1555543245
90466.53474828327
89042.79299979609
87788.44318811042
86671.02673957073
85666.21469578656
84755.35863200181
83923.88973312645
83160.23737513927
82455.0783454794
81800.80336483392
81191.1303482447
80620.81907925931
80085.45739372798
79581.29868561418
79105.13683010446
78654.20877677531
78226.11787039216
77818.77288443455
77430.33909696902
77059.19869085017
76703.9184422037
76363.22315730723
76035.97368107313
75721.14857023943
75417.82872594058
75125.18443309901
74842.4643700617
74568.98624120097
74304.12875450161
74047.32471977978
73798.05508575324
73555.84376742064
73320.25314209097
73090.88011349276
72867.35266063845
72649.3268020753
72436.48391736683
72228.52837695656
72025.1854391334
71826.19937918284
71631.33182087274
71440.36024494423
71253.07665278034
71069.2863664588
70888.80694922632
70711.46723211402
70537.10643488447
70365.573370448

In [71]:
np.random.seed(18)
user1 = np.random.randint(0, len(user_item_array))
while True:
    movie1 = np.random.randint(0, len(user_item_array[user1]))
    if user_item_array[user1][movie1] == 0:
        break
print(f"user {user1} is predicted to have rated {nR[user1][movie1]} ")
top10_user1 = np.argsort(nR[user1])[::-1][:10]

print(f"Top 10 predicted ratings for user {user1}:")

for i in top10_user1:
    print(f"movie {i}: {nR[user1][i]:.1f}")

user 298 is predicted to have rated 3.480942710042128 
Top 10 predicted ratings for user 298:
movie 8157: 6.2
movie 3499: 6.1
movie 8462: 6.1
movie 3936: 6.0
movie 4718: 5.9
movie 3399: 5.9
movie 4153: 5.9
movie 9065: 5.9
movie 5807: 5.9
movie 8150: 5.9


In [72]:
np.random.seed(19)
user2 = np.random.randint(0, len(user_item_array))
while True:
    movie2 = np.random.randint(0, len(user_item_array[user2]))
    if user_item_array[user2][movie2] == 0:
        break
print(f"user {user2} is predicted to have rated {nR[user2][movie2]} ")
top10_user2 = np.argsort(nR[user2])[::-1][:10]

print(f"Top 10 predicted ratings for user {user2}:")

for i in top10_user2:
    print(f"movie {i}: {nR[user2][i]:.1f}")

user 605 is predicted to have rated 4.995576774816088 
Top 10 predicted ratings for user 605:
movie 8796: 6.3
movie 2520: 6.0
movie 4153: 6.0
movie 6937: 5.9
movie 3399: 5.9
movie 3394: 5.9
movie 4996: 5.8
movie 4043: 5.8
movie 5510: 5.8
movie 4866: 5.8


In [74]:
np.random.seed(20)
user3 = np.random.randint(0, len(user_item_array))
while True:
    movie3 = np.random.randint(0, len(user_item_array[user3]))
    if user_item_array[user3][movie3] == 0:
        break
print(f"user {user3} is predicted to have rated {nR[user3][movie3]} ")
top10_user3 = np.argsort(nR[user3])[::-1][:10]

print(f"Top 10 predicted ratings for user {user3}:")

for i in top10_user3:
    print(f"movie {i}: {nR[user3][i]:.1f}")

user 355 is predicted to have rated 4.60079478357899 
Top 10 predicted ratings for user 355:
movie 1098: 5.9
movie 1819: 5.8
movie 6446: 5.7
movie 3980: 5.7
movie 4996: 5.7
movie 1519: 5.7
movie 966: 5.6
movie 7878: 5.6
movie 6129: 5.5
movie 1844: 5.4


In [68]:
for idx in remove_indices:
    y_true = user_item_array[idx]
    y_pred = nR[idx]
    for i in range(len(user_item_array_validation[idx])):
        if user_item_array[idx][i] > 0:
            print(f"User {idx} predicted ratings: {nR[idx][i]}")
            print(f"User {idx} original ratings: {user_item_array[idx][i]}")

User 655 predicted ratings: 3.221258676600623
User 655 original ratings: 4.0
User 655 predicted ratings: 3.8876098555332286
User 655 original ratings: 5.0
User 655 predicted ratings: 3.881719587747197
User 655 original ratings: 5.0
User 655 predicted ratings: 3.9947118005399638
User 655 original ratings: 5.0
User 655 predicted ratings: 3.234619977629675
User 655 original ratings: 4.0
User 655 predicted ratings: 3.6303762810387807
User 655 original ratings: 5.0
User 655 predicted ratings: 3.8659807127672896
User 655 original ratings: 5.0
User 655 predicted ratings: 3.5311936820305276
User 655 original ratings: 5.0
User 655 predicted ratings: 3.8352250335395355
User 655 original ratings: 5.0
User 655 predicted ratings: 3.6590151963027195
User 655 original ratings: 3.0
User 655 predicted ratings: 4.1492458030074575
User 655 original ratings: 5.0
User 655 predicted ratings: 3.7451325579313557
User 655 original ratings: 5.0
User 655 predicted ratings: 4.105301937823512
User 655 original rat

In [42]:
print(len(nR[0]))
print(nR[1][9])

9066
3.9705533625813563


In [69]:
answer = 0
for idx in remove_indices:
    partial_answer = 0
    y_true = user_item_array[idx]
    y_true_part = []
    y_pred = nR[idx]
    y_pred_part = []
    for i in range(len(user_item_array[idx])):
        if user_item_array[idx][i] > 0:
            y_true_part.append(user_item_array[idx][i])
            y_pred_part.append(nR[idx][i])
            # print(f"User {idx} predicted ratings: {nR[idx][i]}")
            # print(f"User {idx} original ratings: {user_item_array[idx][i]}")
    partial_answer = np.mean((np.array(y_true_part) - np.array(y_pred_part))**2)
    print(f"User {idx} MSE: {partial_answer}")
    answer += partial_answer
answer /= len(remove_indices)
print(f"Mean Squared Error: {answer}")

User 655 MSE: 1.6496546399548127
User 189 MSE: 0.9192389617562994
User 58 MSE: 2.769027894356347
User 579 MSE: 0.6346605978519606
User 151 MSE: 0.9770276153190368
User 232 MSE: 0.2805062615656095
User 586 MSE: 1.0763837267890541
User 326 MSE: 0.8084660053659183
User 192 MSE: 1.2403186458161437
User 598 MSE: 1.6086786316436097
User 286 MSE: 5.71807298422549
User 54 MSE: 1.0232326631096187
User 505 MSE: 0.7905454032488121
User 310 MSE: 2.3408258769211012
User 429 MSE: 1.1610678412689228
User 204 MSE: 1.434414359684188
User 327 MSE: 0.6914464772795047
User 401 MSE: 0.9368358517545473
User 593 MSE: 0.8160455990427327
User 555 MSE: 1.0392919403413412
User 436 MSE: 0.8854901314245565
User 518 MSE: 0.6043472659569
User 583 MSE: 2.279697006634141
User 68 MSE: 1.0180510347281349
User 112 MSE: 2.551032201604468
User 534 MSE: 1.8780163297415962
User 193 MSE: 1.1455570531465396
User 201 MSE: 1.6372957325623725
User 180 MSE: 1.0990863870335243
User 277 MSE: 2.227204757029994
User 354 MSE: 0.8967824

In [70]:
answer = 0
for idx in remove_indices:
    y_true = user_item_array[idx]
    y_pred = nR[idx]
    print(y_true)
    print(y_pred)
    answer += np.mean((y_true - y_pred)**2)
print(f"Total MSE: {answer}")
print(f"Number of removed users: {len(remove_indices)}")
answer /= len(remove_indices)
print(f"Mean Squared Error: {answer}")

[0. 0. 0. ... 0. 0. 0.]
[3.88968753 3.71897338 3.02653296 ... 4.24867704 3.2932602  5.0388366 ]
[0. 0. 0. ... 0. 0. 0.]
[3.28142425 2.54821187 2.37942382 ... 3.62019833 2.74324253 5.05316656]
[0. 0. 0. ... 0. 0. 0.]
[4.4514868  3.98922834 3.71263644 ... 5.04873339 2.9846361  5.06977012]
[4.  3.5 0.  ... 0.  0.  0. ]
[3.18593824 2.79948376 3.70939892 ... 3.25303483 2.14612738 3.94253694]
[3.5 0.  0.  ... 0.  0.  0. ]
[3.49281741 3.08405906 2.69677429 ... 4.12598009 3.4871753  4.62661817]
[0. 0. 3. ... 0. 0. 0.]
[3.73474649 2.92498021 2.98712325 ... 3.90316974 2.99247984 4.97903819]
[0. 0. 0. ... 0. 0. 0.]
[3.37848609 2.98280482 2.24720171 ... 3.96836266 2.72522643 4.1232167 ]
[0. 0. 0. ... 0. 0. 0.]
[3.41940485 3.68543892 3.18494968 ... 4.51955624 3.22945923 5.03496799]
[4. 0. 0. ... 0. 0. 0.]
[4.71980242 4.25702945 3.20891332 ... 5.57757752 3.83797811 6.52288542]
[0. 0. 0. ... 0. 0. 0.]
[3.38085646 2.93873515 2.92838271 ... 3.76864917 2.49660149 4.71240769]
[5. 5. 0. ... 0. 0. 0.]
[2.3

evaluate 2

In [ ]:
num_to_remove = int(len(user_item_array) * 0.1)
# Randomly select indices to remove
remove_indices = np.random.choice(len(user_item_array), num_to_remove, replace=False)
# print(f"Indices to remove: {remove_indices}")

for idx in remove_indices:
    for i in range(len(user_item_array_validation[idx])):
        user_item_array_validation[idx][i] = 0

answer = 0
for idx in remove_indices:
    partial_answer = 0
    y_true = user_item_array[idx]
    y_true_part = []
    y_pred = nR[idx]
    y_pred_part = []
    for i in range(len(user_item_array[idx])):
        if user_item_array[idx][i] > 0:
            y_true_part.append(user_item_array[idx][i])
            y_pred_part.append(nR[idx][i])
            # print(f"User {idx} predicted ratings: {nR[idx][i]}")
            # print(f"User {idx} original ratings: {user_item_array[idx][i]}")
    partial_answer = np.mean((np.array(y_true_part) - np.array(y_pred_part))**2)
    print(f"User {idx} MSE: {partial_answer}")
    answer += partial_answer
answer /= len(remove_indices)
print(f"Mean Squared Error: {answer}")

relevant = []
for idx in remove_indices:
    relevant_user = []
    for i in range(len(user_item_array[idx])):
        if user_item_array[idx][i] >= 4.0:
            relevant_user.append(i)
    relevant.append(relevant_user)

## References

https://medium.com/data-science/recommendation-system-matrix-factorization-d61978660b4b